# Data Scraping with Python
```
Course: ENV 700 - Environmental Data Exploration
Authors: John Fay & Luana Lima
```


## Objectives
1. Acquire and scrape data from web sources.
2. Process web-scraped data into reproducible formats.
3. Use functions and iteration to automate scraping processes.

>This version uses `requests` to retrieve HTML, `BeautifulSoup` to select elements from the HTML, `pandas` to organize scraped values, and `seaborn` for plotting.

## What does it mean to scrape data from the web?
The internet is a vast source of data. Some sites provide direct downloads or APIs - which we'll discuss later; others display useful data in webpages without providing a convenient machine-readable download. **Web scraping** means programmatically retrieving those webpages and extracting selected content from their HTML.  

Copy-and-paste is a brute-force form of scraping. A more reproducible approach is to inspect the HTML structure and use tags, classes, IDs, and CSS selectors to target the elements containing the data we need.

>**⚠️Important caveats**  
>Not every site permits or welcomes automated scraping. Before scraping, check the site's terms, `robots.txt` where relevant, and any published API or download options. Keep requests modest, identify your client when appropriate, and pause between repeated requests.
>
> **For discussion:** What are examples of ethical and unethical web scraping?

## Setup: Python packages used for scraping

In [ ]:
#Pandas
import pandas as pd

#Web scraping packages
import requests
from bs4 import BeautifulSoup

#Automation tools
import time
from itertools import product

#Plotting
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.nonparametric.smoothers_lowess import lowess
sns.set_theme(style="darkgrid", context='notebook')

## 1.1 Explore the data we want to scrape

- Navigate to: <https://www.ncwater.org/WUDC/app/WWATR/report>

- Then view the 2020 report for facility `0004-0001` (Plant 15):  
<https://www.ncwater.org/WUDC/app/WWATR/report/view/0004-0001/2020>

- We want to collect the **registrant**, **facility name**, **facility type**, and **monthly water-withdrawal values**.

## 1.2 Fetch the webpage

The Python `requests` package can read contents of a web page into memory. Then, we can use `BeautifulSoup()` function imported from the `bs4` package to *parse* and extract specific bits from this ingested web page. 

1. `requests.get()` retrieves the webpage.
2. `BeautifulSoup()` parses the returned HTML.

In [ ]:
url = "https://www.ncwater.org/WUDC/app/WWATR/report/view/0004-0001/2020"

response = requests.get(url)
response.raise_for_status()

webpage = BeautifulSoup(response.text, "html.parser")
type(webpage)

if webpage.title: 
    print(webpage.title.get_text(strip=True))
else: print("No page title")

### Understanding webpages from a coding standpoint
A typical HTML document contains `<head>` and `<body>` sections. HTML is organized with **tags** and attributes such as `id` and `class`. Elements are nested hierarchically—for example, a `<table>` contains rows (`<tr>`), which contain cells (`<td>` or `<th>`).

In your browser, view the page source (`Ctrl+U` on many browsers). Can you locate the registrant, **American & Efird, Inc.**?

Browser developer tools or a CSS-selector helper such as [SelectorGadget](https://chromewebstore.google.com/detail/mhjhnkcfbdhnjickkkdbjoemdmbfginb) can help identify selectors for specific elements.

>*⚠️The **SelectorGadget** only works reliably using the **Google Chrome browser** ⚠️*

## 1.3 Scraping the data
Say we want to scrape some data shown in the page into our Python coding environment. The bits we want to scrape are:
* The name of the Registrant: "`American & Efird, Inc.`" 
* The name of the facility: "`Plant 15`"
* The facility type: "`Industrial`"
* And the average water withdrawals for each month: "`0.584`, `0.647`, `0.543`, etc."

To do this we first need to install a tool on our web browser to be able to call the web text we need. The tool is called a Selector Gadget, which for Chrome can be found [here](https://chrome.google.com/webstore/detail/selectorgadget/mhjhnkcfbdhnjickkkdbjoemdmbfginb?hl=en). 

The selector gadget tool is useful in identifying the internal ids of the elements shown in the web page that we want to extract. Using it is a bit clumsy, often requiring a bit of mucking about, but it still makes our work a lot easier. We'll begin by using it to determine the id of the box containing the registrant of our our site. 
1. Activate the Selector Gadget tool.
2. Click on the box listing the registrant ("American & Efird, Inc."). You'll see a number of boxes in yellow, but we just want the one box in yellow.
3. Click various other boxes until only the one we want is highlighted in yellow (or green). It's ok if others are in red. This may take a bit of trial and error, and if you get too mess up, you can start over by clicking the `Clear` button in the Selector Gadget floating toolbar.

When successful, the box should display `.table tr:nth-child(1) td:nth-child(2)`. This uniquely identifies the html element containing the data we want. 

From there, we use BeautifulSoup to select and extract the data associate with that tag. 
* BeautifulSoup's `.select()` method accepts CSS selectors. 
* `.get_text(strip=True)` extracts the visible text from a selected element.


In [ ]:
registrant_nodes = webpage.select(".table tr:nth-child(1) td:nth-child(2)")
the_registrant = registrant_nodes[0].get_text(strip=True)
the_registrant

### ✅Exercise: facility name and facility type
Use the SelectorGadget to identify the tags associated with the **facility name** and **facility type**.

- Facility Name: `tr:nth-child(2) th+ .left:nth-child(2)`
- Facility Type: `tr:nth-child(2) .left~ .left+ td.left`

In [ ]:
the_facility_name = webpage.select_one(
    "tr:nth-child(2) th+ .left:nth-child(2)"
).get_text(strip=True)

the_facility_type = webpage.select_one(
    "tr:nth-child(2) .left~ .left+ td.left"
).get_text(strip=True)

print(the_facility_name)
print(the_facility_type)

Next, extract all monthly average and maximum daily withdrawal values. Because `.select()` can return multiple elements, a [list comprehension](https://www.w3schools.com/python/python_lists_comprehension.asp) is a natural Python equivalent to `html_nodes(...) |> html_text()`.

In [ ]:
avg_nodes = webpage.select(
    ".table:nth-child(7) td:nth-child(7), .table:nth-child(7) td:nth-child(3)"
)

avg_withdrawals = [node.get_text(strip=True) for node in avg_nodes]
avg_withdrawals

In [ ]:
# Exercise solution: maximum daily withdrawals


## 1.4 Construct a DataFrame and plot the values

The webpage returns the monthly values in a nonchronological order, so we explicitly assign the corresponding month numbers before sorting by date.

In [ ]:
month_order = [1, 7, 2, 8, 3, 9, 4, 10, 5, 11, 6, 12]

df_withdrawals = pd.DataFrame({
    "Month": month_order,
    "Year": 2020,
    "Avg_Withdrawals_mgd": pd.to_numeric(avg_withdrawals),
    "Max_Withdrawals_mgd": pd.to_numeric(max_withdrawals),
})

df_withdrawals = df_withdrawals.assign(
    Registrant=the_registrant,
    Facility_name=the_facility_name,
    Facility_type=the_facility_type,
    Date=lambda d: pd.to_datetime(dict(year=d["Year"], month=d["Month"], day=1)),
).sort_values("Date")

df_withdrawals

In [ ]:
g = sns.relplot(
    data=df_withdrawals,
    x="Date",
    y="Avg_Withdrawals_mgd",
    kind="line",
    marker="o",
    aspect=2
)
g.set(
    title=f"2020 Water usage data for {the_registrant}\n{the_facility_name}",
    xlabel="Date",
    ylabel="Withdrawal (mgd)",
);

# Part 2. Automating the scraping process

## 2.1 Streamline the process
Move the facility ID, year, URL, and selectors into variables. This makes the workflow easier to reuse before we package it as a function.

In [ ]:
#Set variables for the facility ID and the year
the_facility = "0004-0001"
the_year = 2020

#Construct the url using the variables set above
the_base_url = "https://www.ncwater.org/WUDC/app/WWATR/report/view"
the_scrape_url = f"{the_base_url}/{the_facility}/{the_year}"
print(the_scrape_url)

#With the URL set, fetch the web site's contents
response = requests.get(the_scrape_url)
response.raise_for_status()
the_website = BeautifulSoup(response.text, "html.parser")

#The tags remain the same no matter what facility or year you fetch
the_registrant_tag = ".table tr:nth-child(1) td:nth-child(2)"
the_facility_name_tag = "tr:nth-child(2) th+ .left:nth-child(2)"
the_facility_type_tag = "tr:nth-child(2) .left~ .left+ td.left"
the_data_tag = ".table:nth-child(7) td:nth-child(7), .table:nth-child(7) td:nth-child(3)"

#Extract the values from the fetch websites
the_registrant = the_website.select_one(the_registrant_tag).get_text(strip=True)
the_facility_name = the_website.select_one(the_facility_name_tag).get_text(strip=True)
the_facility_type = the_website.select_one(the_facility_type_tag).get_text(strip=True)
the_withdrawals = [x.get_text(strip=True) for x in the_website.select(the_data_tag)]

#Process the values into a dataframe
month_order = [1, 7, 2, 8, 3, 9, 4, 10, 5, 11, 6, 12]
df_withdrawals = pd.DataFrame({
    "Month": month_order,
    "Year": the_year,
    "Avg_Withdrawals_mgd": pd.to_numeric(the_withdrawals),
}).assign(
    Registrant=the_registrant,
    Facility_name=the_facility_name,
    Facility_type=the_facility_type,
)
df_withdrawals["Date"] = pd.to_datetime(
    dict(year=df_withdrawals["Year"], month=df_withdrawals["Month"], day=1)
)
df_withdrawals = df_withdrawals.sort_values("Date")

#Display the results
df_withdrawals.head()

Re-run the workflow with `the_year = 2015`. Then try `facility ID = '0218-0238'`. This demonstrates why parameterizing the workflow is useful.

## 2.2 Automation, Step 1: Build a function

A function lets us provide a year and facility ID and receive a clean DataFrame in return.

In [ ]:
def scrape_withdrawals(the_year, the_facility, pause=0):
    """Scrape monthly average withdrawals for one NC facility and year."""
    #Construct the URL and fetch the web page
    url = f"https://www.ncwater.org/WUDC/app/WWATR/report/view/{the_facility}/{the_year}"
    response = requests.get(url)
    response.raise_for_status()
    the_website = BeautifulSoup(response.text, "html.parser")

    #Extract the data from the website object
    registrant = the_website.select_one(".table tr:nth-child(1) td:nth-child(2)").get_text(strip=True)
    facility_name = the_website.select_one("tr:nth-child(2) th+ .left:nth-child(2)").get_text(strip=True)
    facility_type = the_website.select_one("tr:nth-child(2) .left~ .left+ td.left").get_text(strip=True)
    withdrawals = [x.get_text(strip=True) for x in the_website.select(".table:nth-child(7) td:nth-child(7), .table:nth-child(7) td:nth-child(3)")]

    #Wrangle into a dataframe
    month_order = [1, 7, 2, 8, 3, 9, 4, 10, 5, 11, 6, 12]
    df = pd.DataFrame({
        "Month": month_order,
        "Year": the_year,
        "Avg_Withdrawals_mgd": pd.to_numeric(withdrawals),
        "Registrant": registrant,
        "Facility_name": facility_name,
        "Facility_type": facility_type,
    })
    df["Date"] = pd.to_datetime(dict(year=df["Year"], month=df["Month"], day=1))
    df = df.sort_values("Date").reset_index(drop=True)

    #Pause before proceeding
    if pause: time.sleep(pause)

    #Return the dataframe
    return df

Use the function to scrape 2025 data from facility "0004-0001", the plot the results.

In [ ]:
the_df = scrape_withdrawals(2024, "0004-0001")
the_df.plot(kind='line',x='Date',y='Avg_Withdrawals_mgd',figsize=(10,4),marker='o');

## 2.3 Iterate across multiple years

Python list comprehensions provide a compact way to repeat a function over a sequence. `pd.concat()` combines the returned DataFrames.

In [ ]:
#Set the years we want to scrape, and the facility
the_years = range(2023, 2025)
my_facility = "0004-0001"

#Use list comprehension to capture the result of applying the function to our range of years
the_dfs = [scrape_withdrawals(year, my_facility, pause=1) for year in the_years]

#Combine the resulting dataframes into a single one
the_df = pd.concat(the_dfs, ignore_index=True)

Plot the results:

In [ ]:
#Extract values for plotting
the_registratnt = the_df.loc[0,'Registrant']
the_facility_name = the_df.loc[0,'Facility_name']
start_date = the_df['Date'].min()
end_date = the_df['Date'].max()

#Create the facetgrid 
g = sns.relplot(
    data=the_df,
    x="Date",
    y="Avg_Withdrawals_mgd",
    kind="line",
    aspect=3,
    marker='o'
)

#Set the axis labels and the limits of the x-axis
g.set(
    xlabel="Date", 
    ylabel="Withdrawal (mgd)",
    #xlim=(start_date, end_date),
    #ylim=(0,the_df['Avg_Withdrawals_mgd'].max()*1.1)
)

#Set titles: "suptitle" is a larger "super" title
g.figure.suptitle(
    f"{the_registrant} - {the_facility_name}",
    fontweight="bold",
    fontsize=14,
    y=1.05 #Place it a bit higher, to make space for the title below
)

#Title is a smaller title
g.ax.set_title(
    f"Water usage data: {start_date.year}-{end_date.year}",
    fontsize=11,
    pad = 2
);

# Part 3. Web crawling

Web *crawling* extends scraping across linked pages. Here, the [main report page](https://www.ncwater.org/WUDC/app/WWATR/report) contains links to individual facility reports. We first collect the facility IDs from those links and then apply our scraping function to selected facilities.

## 3.1 Get a list of facility IDs

The R lab selects `#content a`, extracts each link's `href`, splits the path, and takes the facility ID. In Python, we can use `urljoin()` and string methods.

In [ ]:
# Fetch the contents of the page listing facilities 
the_main_url = "https://www.ncwater.org/WUDC/app/WWATR/report"
response = requests.get(the_main_url)
response.raise_for_status()
the_main_website = BeautifulSoup(response.text, "html.parser")

# Extract the facility ID URLs (href) into a list
links = the_main_website.select("#content a")
hrefs = [link.get("href") for link in links if link.get("href")]

# Extract the facility IDs from the URLs
the_facility_ids = []
for href in hrefs:
    #Split into components
    href_objects = href.split('/')
    #Get the facility ID (2nd last)
    facility_id = href_objects[-2]
    #Append to list
    the_facility_ids.append(facility_id)

#--Or--using list comprehension
the_facility_ids = [href_objects.split('/')[-2] for href_objects in hrefs]

#Show the first 10 items in the list
the_facility_ids[:10]

## 3.2 Scrape 2020 data for a subset of sites

To avoid placing unnecessary demand on the server, use a small subset and pause between requests: we'll exctract data for 10 sites for the year 2020 and produce a box plot of the distribution of withdrawals by facility type. 

In [ ]:
# Use the first 20 facilities rather than a random sample for reproducibility.
facility_subset = the_facility_ids[:10]

dfs_year = [
    scrape_withdrawals(2020, facility_id, pause=1)
    for facility_id in facility_subset
]
df_year = pd.concat(dfs_year, ignore_index=True)


sns.catplot(
    data=df_year,
    x="Facility_type",
    y="Avg_Withdrawals_mgd",
    kind="box",
    height=5,
    aspect=1.5,
).set_xticklabels(rotation=45);

## 3.3 Scrape across years and sites

`itertools.product()` creates every combination of year and facility ID—the Python equivalent of creating a parameter grid before mapping the scraping function across it.

In [ ]:
#Demo of `itertools.product()`
the_facilities = the_facility_ids[-3:]
for facility in the_facilities:
    for year in the_years: 
        print (year, facility)
# is the same as 
for year, facility in product(the_years, the_facilities):
    print(year, facility)

In [ ]:
the_facilities = the_facility_ids[-3:]
the_years = [2023, 2024, 2025]

frames = [
    scrape_withdrawals(year, facility, pause=1)
    for year, facility in product(the_years, the_facilities)
]
the_df = pd.concat(frames, ignore_index=True)

sns.relplot(
    data=the_df,
    x="Date",
    y="Avg_Withdrawals_mgd",
    hue="Facility_name",
    kind="line",
    height=5,
    aspect=1.5,
).set(xlabel="Date", ylabel="Withdrawal (mgd)");

## Takeaways

- Web scraping starts by retrieving and parsing HTML.
- CSS selectors let us target specific webpage elements.
- Scraped text usually needs to be converted and reorganized before analysis.
- Functions make a scraping workflow reusable.
- Iteration allows the same workflow to operate across years, facilities, or combinations of parameters.
- Automated scraping should be conservative and respectful of the source server.